In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system_prompt=None, stop_sequences=None):
    parameters = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if system_prompt:
        parameters["system"] = system_prompt

    if stop_sequences:
        parameters["stop_sequences"] = stop_sequences

    message = client.messages.create(**parameters)
    return message.content[0].text if getattr(message.content[0], 'text', None) else message.content[1].text

In [3]:
import json
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "python" or "json" or "regex"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [4]:
dataset = generate_dataset()
print(dataset)

[{'task': "Parse an AWS S3 bucket name and key from an S3 URI in the format 's3://bucket-name/path/to/key'", 'format': 'regex'}, {'task': 'Create a JSON CloudFormation template snippet that defines an IAM role with a trust policy allowing EC2 instances to assume it', 'format': 'json'}, {'task': 'Write a Python function that validates an AWS ARN string and returns True if valid, False otherwise', 'format': 'python'}]


In [5]:
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [17]:
def grade_by_model(test_case, output):
    try:
        # Create evaluation prompt
        eval_prompt = f"""
        You are an expert code reviewer. Evaluate this AI-generated solution.
        
        Task: {test_case}
        Solution: {output}
        
        Provide your evaluation as a structured JSON object with:
        - "strengths": An array of 1-3 key strengths
        - "weaknesses": An array of 1-3 key areas for improvement  
        - "reasoning": A concise explanation of your assessment
        - "score": A number between 1-10
        """
        
        messages = []
        add_user_message(messages, eval_prompt)
        add_assistant_message(messages, "```json")
        
        eval_text = chat(messages, stop_sequences=["```"])
        return json.loads(eval_text)
    except Exception as e:
        print(f"Error in grade_by_model: {e} - eval_text: {eval_text}")

In [18]:
import ast
import re

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(output, test_case):
    format = test_case.get("format")
    if format == "json":
        return validate_json(output)
    elif format == "python":
        return validate_python(output)
    elif format == "regex":
        return validate_regex(output)
    else:
        return 0

In [19]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [25]:
def run_test_case(test_case):
    output = run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    syntax_score = grade_syntax(output, test_case)
    reasoning = model_grade["reasoning"]
    score = (model_score + syntax_score) / 2
    
    return {
        "output": output, 
        "test_case": test_case, 
        "model_score": model_score,
        "syntax_score": syntax_score,
        "score": score,
        "reasoning": reasoning
    }

In [26]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

In [27]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 6.666666666666667


In [28]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nfunction parseS3Uri(s3Uri) {\n  // Match the pattern s3://bucket-name/path/to/key\n  const match = s3Uri.match(/^s3:\\/\\/([^\\/]+)\\/(.+)$/);\n  \n  if (!match) {\n    throw new Error(`Invalid S3 URI format: ${s3Uri}`);\n  }\n  \n  return {\n    bucket: match[1],\n    key: match[2]\n  };\n}\n\n// Test cases\nconsole.log(parseS3Uri('s3://my-bucket/path/to/key'));\n// Output: { bucket: 'my-bucket', key: 'path/to/key' }\n\nconsole.log(parseS3Uri('s3://bucket-name/file.txt'));\n// Output: { bucket: 'bucket-name', key: 'file.txt' }\n\nconsole.log(parseS3Uri('s3://another-bucket/deep/nested/path/to/file.json'));\n// Output: { bucket: 'another-bucket', key: 'deep/nested/path/to/file.json' }\n\n// Error case\ntry {\n  parseS3Uri('invalid-uri');\n} catch (error) {\n  console.log(error.message);\n  // Output: Invalid S3 URI format: invalid-uri\n}\n",
    "test_case": {
      "task": "Parse an AWS S3 bucket name and key from an S3 URI in the format 's3://bucket-name/path/t